# 1단계: 글자(Character) 단위 토크나이저

## 이 노트북에서 배우는 것

컴퓨터는 글자를 직접 이해하지 못합니다. 모든 텍스트는 결국 **숫자**로 변환되어야 합니다.  
이 과정을 **토크나이징(Tokenizing)** 이라고 부릅니다.

이 노트북을 마치면 다음을 이해하게 됩니다:

| 개념 | 설명 |
|------|------|
| **어휘집 (Vocabulary)** | 토크나이저가 아는 모든 토큰의 목록 |
| **인코딩 (Encoding)** | 텍스트 → 숫자 목록 |
| **디코딩 (Decoding)** | 숫자 목록 → 텍스트 |
| **특수 토큰 (Special Tokens)** | PAD, UNK, BOS, EOS |
| **OOV 문제** | Out-Of-Vocabulary, 어휘집에 없는 단어 |

> **학습 방법**: 각 셀을 직접 실행하면서 결과를 확인하세요. 숫자를 바꿔보고, 텍스트를 바꿔보면서 동작을 이해하는 것이 중요합니다.

---
## 1단계: 컴퓨터는 숫자만 이해한다

컴퓨터 안에서 글자 `'A'`는 사실 숫자 `65`입니다.  
이 숫자 체계를 **Unicode 코드포인트**라고 부릅니다.

파이썬에서는 두 가지 함수로 이를 확인할 수 있습니다:
- `ord(글자)` → 글자를 숫자로 변환
- `chr(숫자)` → 숫자를 글자로 변환

In [ ]:
# ord()는 글자를 Unicode 코드포인트(정수)로 변환합니다
print(ord('h'))  # 104
print(ord('e'))  # 101
print(chr(104))  # 'h'

# "hello"를 숫자 목록으로 변환해보기
text = "hello"
numbers = [ord(c) for c in text]
print(numbers)

---
## 2단계: 직접 어휘집 만들기 (클래스 없이)

**어휘집(Vocabulary)** 이란 토크나이저가 알고 있는 모든 토큰과 그 번호의 목록입니다.

글자 단위 토크나이저를 직접 손으로 만들어 봅시다:
1. 텍스트에서 고유한 글자를 모두 수집합니다
2. 각 글자에 고유한 번호(ID)를 부여합니다
3. `글자 → ID` 와 `ID → 글자` 두 방향의 딕셔너리를 만듭니다

In [ ]:
# 텍스트에서 고유 글자 수집
text = "hello world"
unique_chars = sorted(set(text))  # set()으로 중복 제거, sorted()로 정렬
print("고유 글자들:", unique_chars)

# 각 글자에 번호 부여
char_to_id = {}  # 글자 → 숫자 (인코딩에 사용)
id_to_char = {}  # 숫자 → 글자 (디코딩에 사용)
for i, char in enumerate(unique_chars):
    char_to_id[char] = i
    id_to_char[i] = char

print("어휘집:", char_to_id)
print("vocab_size:", len(char_to_id))

---
## 3단계: 인코딩 직접 구현

**인코딩(Encoding)** 이란 텍스트의 각 글자를 어휘집에서 찾아 해당 ID로 바꾸는 과정입니다.

```
"hello" → [h, e, l, l, o] → [ID(h), ID(e), ID(l), ID(l), ID(o)]
```

In [ ]:
# 인코딩 함수 직접 작성
def my_encode(text, char_to_id):
    # 텍스트의 각 글자를 ID로 변환
    return [char_to_id[c] for c in text]

encoded = my_encode("hello", char_to_id)
print("인코딩 결과:", encoded)

---
## 4단계: 디코딩 직접 구현

**디코딩(Decoding)** 이란 숫자 목록을 다시 텍스트로 복원하는 과정입니다.  
인코딩의 반대 방향입니다.

```
[ID(h), ID(e), ID(l), ID(l), ID(o)] → [h, e, l, l, o] → "hello"
```

In [ ]:
# 디코딩 함수 직접 작성
def my_decode(ids, id_to_char):
    # 각 ID를 글자로 변환한 뒤 하나의 문자열로 합침
    return ''.join([id_to_char[i] for i in ids])

decoded = my_decode(encoded, id_to_char)
print("디코딩 결과:", decoded)

# 원본과 동일한지 검증
assert decoded == "hello"
print("원본 복원 성공!")

---
## 5단계: OOV 문제 직접 경험하기

**OOV(Out-Of-Vocabulary)** 란 어휘집에 없는 글자나 단어를 만났을 때 발생하는 문제입니다.

우리가 만든 어휘집은 `"hello world"` 라는 텍스트로만 학습했습니다.  
만약 어휘집에 없는 글자를 인코딩하려 하면 어떻게 될까요?

In [ ]:
# 'z'는 학습 텍스트("hello world")에 없었습니다
try:
    my_encode("z", char_to_id)
except KeyError as e:
    print(f"OOV 발생! {e}는 어휘집에 없습니다")
    print("해결책 → UNK 토큰")

---
## 6단계: 특수 토큰이란? (PAD, UNK, BOS, EOS)

실제 언어 모델에서는 일반 글자 외에 **특수 토큰(Special Tokens)** 이 필요합니다:

| 토큰 | 이름 | 역할 |
|------|------|------|
| `[PAD]` | Padding | 여러 문장의 길이를 맞추기 위한 빈 자리 채우기 |
| `[UNK]` | Unknown | OOV 글자를 대신하는 미지 토큰 |
| `[BOS]` | Begin Of Sequence | 문장의 시작을 알리는 토큰 |
| `[EOS]` | End Of Sequence | 문장의 끝을 알리는 토큰 |

특수 토큰을 **먼저** 0번부터 등록하는 이유:  
모델 코드 전반에서 `PAD=0`, `UNK=1` 을 상수처럼 사용하기 때문입니다.

In [ ]:
# 특수 토큰을 먼저 0번부터 등록해야 하는 이유:
# 모델 코드 전반에서 PAD=0, UNK=1을 상수로 사용하기 때문
special_tokens = ["[PAD]", "[UNK]", "[BOS]", "[EOS]"]

char_to_id_v2 = {}
id_to_char_v2 = {}

# 특수 토큰 먼저 (0번부터)
for i, tok in enumerate(special_tokens):
    char_to_id_v2[tok] = i
    id_to_char_v2[i] = tok

# 일반 글자는 4번부터
for char in sorted(set("hello world")):
    idx = len(char_to_id_v2)
    char_to_id_v2[char] = idx
    id_to_char_v2[idx] = char

print("어휘집 (특수토큰 포함):", char_to_id_v2)

# BOS/EOS를 붙여서 인코딩
# BOS=2, 글자 ID들, EOS=3
encoded_with_special = [2] + [char_to_id_v2[c] for c in "hello"] + [3]
print("BOS+hello+EOS:", encoded_with_special)

---
## 7단계: 이제 정리된 클래스 코드를 봅시다 — `char_tokenizer.py`

지금까지 손으로 했던 작업들을 `char_tokenizer.py` 파일에서 클래스로 깔끔하게 정리해 두었습니다.

직접 파일을 열어서 코드를 읽어보세요. 낯선 코드가 아니라 방금 한 것들의 정리된 버전입니다.

`char_tokenizer.py`는 위에서 손으로 한 것을 클래스로 정리한 버전입니다.

핵심 메서드:

| 메서드 | 하는 일 |
|--------|----------|
| `train(text)` | 텍스트를 받아 어휘집 구축 |
| `encode(text, add_special_tokens)` | 텍스트 → ID 목록 |
| `decode(ids, skip_special_tokens)` | ID 목록 → 텍스트 |
| `encode_batch(texts, pad_to_max_length)` | 여러 문장을 한 번에 인코딩 + PAD |
| `save(path)` / `load(path)` | 어휘집을 JSON 파일로 저장/불러오기 |

In [ ]:
# 클래스 import 및 실제 사용
import sys, os

# 현재 노트북이 있는 폴더(stage1_char)를 파이썬 경로에 추가
sys.path.insert(0, os.path.dirname(os.path.abspath("__file__")))
from char_tokenizer import CharTokenizer

# 토크나이저 생성 및 학습
tokenizer = CharTokenizer()
tokenizer.train("hello world")
print("어휘집:", tokenizer.get_vocab())

In [ ]:
# 인코딩·디코딩 테스트
encoded = tokenizer.encode("hello", add_special_tokens=True)
print("인코딩:", encoded)  # [BOS] + 글자 IDs + [EOS]

decoded = tokenizer.decode(encoded, skip_special_tokens=True)
print("디코딩:", decoded)  # 특수 토큰 제외하고 복원

In [ ]:
# 배치 인코딩 + PAD 확인
# 세 문장의 길이가 다르지만, PAD로 채워서 동일한 길이로 만들어줍니다
batch = tokenizer.encode_batch(
    ["hello", "hi", "hello world"],
    pad_to_max_length=True
)
for i, ids in enumerate(batch):
    print(f"문장 {i+1}: {ids}")

In [ ]:
# JSON 저장 → 내용 확인
import json

save_path = "tokenizer.json"
tokenizer.save(save_path)

# 저장된 JSON 내용 출력
with open(save_path, encoding="utf-8") as f:
    data = json.load(f)
print("저장된 어휘집:", json.dumps(data, ensure_ascii=False, indent=2))

In [ ]:
# 새 토크나이저로 불러오기 → ID 일치 확인
tok2 = CharTokenizer()
tok2.load(save_path)

ids1 = tokenizer.encode("hello")
ids2 = tok2.encode("hello")
print(f"원본:        {ids1}")
print(f"불러온 것:   {ids2}")
print(f"동일한가? {ids1 == ids2}")

---
## 정리

오늘 배운 내용을 정리합니다:

1. **글자 → 숫자**: `ord()` / `chr()` 로 Unicode 코드포인트 확인
2. **어휘집(Vocabulary)**: 고유 글자에 번호를 부여한 딕셔너리 (`char_to_id`, `id_to_char`)
3. **인코딩**: 텍스트의 각 글자를 ID로 변환
4. **디코딩**: ID 목록을 다시 텍스트로 복원
5. **OOV 문제**: 어휘집에 없는 글자를 만나면 `KeyError` 발생 → `[UNK]` 로 해결
6. **특수 토큰**: `[PAD]=0`, `[UNK]=1`, `[BOS]=2`, `[EOS]=3` — 반드시 먼저 등록
7. **save/load**: JSON 파일로 어휘집을 저장하면 나중에 재사용 가능

---

## 다음 단계 예고: 2단계 — 단어(Word) 단위 토크나이저

글자 단위 토크나이저는 OOV가 없다는 장점이 있지만,  
`"hello"` 를 5개의 토큰으로 쪼개는 비효율이 있습니다.

다음 노트북에서는 **단어 단위** 토크나이저를 만들어보고,  
그 방법이 가진 치명적인 한계(OOV, vocab 폭발)를 직접 경험해 봅니다.